<a href="https://colab.research.google.com/github/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/07_gradio_rakam_tanima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a> &nbsp; <a href="https://github.com/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/07_gradio_rakam_tanima.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-Kaynak-181717?logo=github" alt="GitHub'da Görüntüle"/></a>

> 🚀 **Bu notebook'u tarayıcıda çalıştırmak için sol üstteki "Colab'da Aç" butonuna tıklayın** — hiçbir şey kurmanıza gerek yok!


<div style="background: linear-gradient(135deg, #0B3D91 0%, #1565C0 60%, #1E88E5 100%); padding: 26px 30px; border-radius: 14px; color: white; margin: 0 0 24px 0; font-family: Georgia, serif; box-shadow: 0 4px 14px rgba(11,61,145,0.25);">

<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
  <span style="font-size: 11px; font-weight: bold; letter-spacing: 2px; color: #BBDEFB;">MEB · YEĞİTEK · DERİN ÖĞRENME SERİSİ</span>
  <span style="font-size: 11px; background: rgba(255,255,255,0.18); padding: 5px 12px; border-radius: 20px; color: white; font-weight: bold;">DERS 7 / 9</span>
</div>

<h1 style="margin: 8px 0 4px 0; color: white; font-size: 28px; line-height: 1.2;">✍️ Gradio ile Rakam Tanıma Uygulaması</h1>

<p style="margin: 8px 0 0 0; color: #E3F2FD; font-size: 13px;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong> — Öğretmenler için Uygulamalı Yapay Zeka<br>
<span style="color: #BBDEFB;">⏱️ Süre: ~50 dakika · 🖥️ Ortam: Google Colab veya yerel Python</span>
</p>

<div style="border-top: 1px solid rgba(255,255,255,0.22); margin-top: 14px; padding-top: 12px; font-size: 12px; color: #E3F2FD;">
🎯 Bu notebook <strong>non-teknik kitle</strong> için hazırlanmıştır · Kavramlar günlük hayattan analojilerle anlatılır · Kod minimum, açıklama maksimumdur.
</div>

</div>

## 🎯 Bu Notebook'ta Ne Öğreneceğiz?

Şimdiye kadar modeller eğittik, ama hep notebook içinde kaldı. Bu notebook'ta modelinizi **gerçek bir uygulamaya** dönüştüreceğiz!

Sonunda şunları yapabiliyor olacaksınız:

- 🖼️ Görüntü verisi (MNIST) ile sinir ağı eğitmek
- 💾 **Eğitilmiş modeli dosyaya kaydetmek** (`.keras` dosyası)
- 📂 **Kaydedilmiş modeli geri yüklemek**
- 🌐 **Gradio** ile bir web arayüzü açmak
- 🎨 Tarayıcıda **rakam çiz → tahmin gör** uygulaması yapmak

Bu, gerçek dünyada AI ürünlerinin nasıl çalıştığının tam bir minyatürü!

## 🤔 Önce Bir Senaryo

Postacı düşünün. Türkiye'de PTT, posta kodlarını **el yazısı** olarak okumak zorunda. Tabii artık otomatik makineler yapıyor — ama bu makinelerin arkasında ne var?

Cevap: **Sinir ağları**. Onlara binlerce el yazısı rakam gösterilir, sonra her zarfı saniyeler içinde tanırlar.

Biz de aynısını yapacağız!

---

### 📚 MNIST Veri Seti

MNIST, derin öğrenmenin "Merhaba Dünya"sıdır. İçinde:
- **70.000** adet 28×28 piksel **el yazısı rakam** (0-9) görüntüsü var
- **60.000'i** eğitim, **10.000'i** test için ayrılmış
- Görüntüler **siyah-beyaz** (gri tonlamalı)

Bu veri seti Keras'ta hazır geliyor — tek satır yükleyeceğiz.


## 📦 Kurulum

> ⚠️ **Önemli:** Gradio Colab'da otomatik yüklü değil. Aşağıdaki ilk hücre Gradio'yu yükler — sonra kütüphaneleri import ederiz.

In [ ]:
# Gradio'yu yükle (Colab'da gerekir)
!pip install -q gradio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
import gradio as gr

keras.utils.set_random_seed(42)
print("✅ Hazır! Gradio sürümü:", gr.__version__)

## 📊 Veriyi Tanıyalım

In [ ]:
# MNIST'i yükle (otomatik internetten indirilir, sonra cache'lenir)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"📦 Eğitim verisi: {X_train.shape}  →  {X_train.shape[0]} resim, her biri {X_train.shape[1]}x{X_train.shape[2]} piksel")
print(f"📦 Test verisi  : {X_test.shape}")
print(f"🏷️  Etiketler  : 0-9 arası rakamlar")

### Birkaç Örnek Görelim

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Etiket: {y_train[i]}', fontsize=11)
    ax.axis('off')
plt.suptitle('🔢 MNIST — İlk 10 Eğitim Örneği', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Veri Önişleme (Çok Önemli!)

Sinir ağları **0-1 arasındaki** sayıları sever. Şu an pikseller 0-255 arasında.

Yapacağımız:
1. **Normalize et:** 255'e böl, böylece her piksel 0.0-1.0 arasına gelir.
2. **Bir boyut ekle:** Keras "kaç görüntü, yükseklik, genişlik, kanal" formatı bekler.

> 🎓 **Analoji:** Tariflerde bazen "1 bardak un" yerine "100 gram un" yazar. Aynı bilgi, farklı ölçek. Sinir ağına da en sevdiği ölçekle vermek lazım.


In [ ]:
# 0-255 → 0-1 arasına ölçekle
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

print(f"✅ Normalize edildi. Min: {X_train.min()}, Max: {X_train.max()}")

## 🧠 Modeli Kuralım

Bu kez **görüntü** ile çalışıyoruz ama henüz CNN değil — düz Dense katmanlar yeterli (sıradaki notebook'ta CNN'e geçeceğiz).

Plan:
1. **Reshape:** 28×28'i 28×28×1'e çevir (veri çoğaltma katmanları 4D bekliyor).
2. **Veri çoğaltma:** Eğitim sırasında rakamı hafifçe döndür/yakınlaştır/kaydır → tarayıcıda elle çizilen rakamlara dayanıklı olsun.
3. **Flatten:** Görüntüyü 784 elemanlı düz vektöre çevir.
4. **Dense (128, ReLU):** 128 nöronlu gizli katman.
5. **Dropout (0.2):** Aşırı öğrenmeyi önlemek için %20 nöronu rastgele kapat.
6. **Dense (10, Softmax):** 10 rakam için olasılık çıktısı.

> 🎓 **Veri çoğaltma neden?** MNIST eğitim seti çok temiz — rakamlar hep aynı stilde, aynı boyutta. Sen tarayıcıda farklı çizersin (daha kalın, hafif eğik, biraz kaymış). Bu katmanlar **eğitim sırasında** her rakamı hafifçe değiştirip modele "her türlü çizimi gör" der. Sınava sadece düz soruyla çalışmak vs. soruyu farklı şekillerde çözmek gibi.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Reshape((28, 28, 1)),
    # 🎲 Veri çoğaltma katmanları — sadece eğitim sırasında aktif olur.
    # Modeli farklı şekilde çizilmiş rakamlara karşı (büyüklük, eğim, kayma)
    # daha sağlam yapar. Tarayıcıda elle çizilen rakamları tanımak için ŞART.
    layers.RandomRotation(0.05, fill_mode='constant'),
    layers.RandomZoom(0.10, fill_mode='constant'),
    layers.RandomTranslation(0.05, 0.05, fill_mode='constant'),
    layers.Flatten(),                                 # 28x28x1 → 784 vektör
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')            # 10 sınıf (0-9)
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 🚀 Eğitim

10 epoch — veri çoğaltma var, biraz fazla epoch işe yarar.

In [ ]:
gecmis = model.fit(
    X_train, y_train,
    epochs=10,        # Veri çoğaltma var, biraz daha fazla epoch gerekir
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Eğitim grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(gecmis.history['accuracy'], label='Eğitim', linewidth=2)
axes[0].plot(gecmis.history['val_accuracy'], label='Doğrulama', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Doğruluk')
axes[0].set_title('Doğruluk', fontweight='bold'); axes[0].legend()

axes[1].plot(gecmis.history['loss'], label='Eğitim', linewidth=2)
axes[1].plot(gecmis.history['val_loss'], label='Doğrulama', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Hata')
axes[1].set_title('Hata', fontweight='bold'); axes[1].legend()

plt.tight_layout()
plt.show()

## 🔍 Test Doğruluğu

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"🎯 Test doğruluğu: {test_acc * 100:.2f}%")
print(f"   {len(X_test)} test örneğinden ~{int(test_acc * len(X_test))} tanesi doğru!")

### Birkaç Tahmin Görelim

In [ ]:
tahminler = model.predict(X_test[:10], verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    tahmin = np.argmax(tahminler[i])
    gercek = y_test[i]
    renk = 'green' if tahmin == gercek else 'red'
    ax.imshow(X_test[i], cmap='gray')
    ax.set_title(f'Tahmin: {tahmin} | Gerçek: {gercek}', fontsize=10, color=renk)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 💾 Modeli Kaydedelim

Bu, en önemli adım! Bütün eğitimi yaptık — **kaybetmek istemeyiz**.

Keras modelini tek satırla `.keras` dosyasına kaydediyoruz:

In [ ]:
# Modeli kaydet
model.save('rakam_modeli.keras')

import os
boyut = os.path.getsize('rakam_modeli.keras') / 1024
print(f"✅ Model kaydedildi: rakam_modeli.keras ({boyut:.1f} KB)")
print()
print("Bu dosya artık ağırlıkları + mimariyi içeriyor.")
print("Bilgisayarınızı kapatıp açsanız bile bu modeli yükleyip kullanabilirsiniz.")

## 📂 Modeli Geri Yükleyelim

Şimdi bilgisayarınızı yeni açtınız diyelim. Yapılacak tek şey:

In [ ]:
# Yeni bir değişkene yükle
yuklenen_model = keras.models.load_model('rakam_modeli.keras')

# Aynı tahminleri veriyor mu kontrol et
yeni_tahmin = yuklenen_model.predict(X_test[:1], verbose=0)
print(f"İlk test örneğinin tahmini: {np.argmax(yeni_tahmin)}")
print(f"Gerçek değer: {y_test[0]}")
print()
print("✅ Yüklenen model çalışıyor — tıpkı eğittiğimiz gibi!")

## 🌐 Gradio ile Web Arayüzü

Şimdi en eğlenceli bölüm: **Tarayıcıda rakam çiz → tahmin gör!**

### Gradio Nedir?

Gradio, makine öğrenmesi modellerine **5 dakikada web arayüzü** ekleyen bir kütüphane. Aşağıdaki adımları yapacağız:

1. Tahmin yapan bir Python fonksiyonu yaz
2. Bu fonksiyonu Gradio'ya ver
3. Gradio otomatik olarak web sayfası açsın

> 🎓 **Tarif:** Sen "girişte resim al, çıkışta yazı ver" dersin, Gradio kalanı halleder.


In [ ]:
from PIL import Image as PILImage


def _ciz_to_uint8(cizim):
    """Sketchpad çıktısı → 'rakam=parlak (255), arka plan=koyu (0)' 2D uint8 + debug."""
    if cizim is None:
        return None, "girdi None"

    if isinstance(cizim, dict):
        keys = list(cizim.keys())
        img = cizim.get('composite')
        src = "composite"
        if img is None:
            layer_list = cizim.get('layers') or []
            if layer_list:
                img = layer_list[0]; src = "layers[0]"
        if img is None:
            return None, f"dict.{keys} boş"
    else:
        img = cizim; src = type(cizim).__name__

    arr = np.array(img)
    if arr.size == 0:
        return None, f"{src}: boş"

    info = f"{src} {arr.shape} dtype={arr.dtype}"

    # RGBA / RGB → gri
    if arr.ndim == 3:
        rgb = arr[..., :3] if arr.shape[-1] >= 3 else arr
        gray = rgb.astype('float32').mean(axis=-1)
    else:
        gray = arr.astype('float32')

    # Auto-invert: parlak ortalama (beyaz canvas) ise ters çevir
    if gray.mean() > 127:
        gray = 255.0 - gray
        info += " | invert"

    # Kontrast: en parlak pikseli 255'e çek
    mx = gray.max()
    if mx > 1:
        gray = gray * (255.0 / mx)

    return gray.astype('uint8'), info


def _ciz_to_mnist(raw):
    """Bbox crop + 20px scale + 28x28 center pad (MNIST standardı)."""
    if raw is None:
        return None
    mask = raw > 30
    if not mask.any():
        return None
    rows = np.where(np.any(mask, axis=1))[0]
    cols = np.where(np.any(mask, axis=0))[0]
    crop = raw[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
    h, w = crop.shape
    if h >= w:
        new_h, new_w = 20, max(1, int(round(w * 20 / h)))
    else:
        new_w, new_h = 20, max(1, int(round(h * 20 / w)))
    pil = PILImage.fromarray(crop.astype('uint8'))
    resized = np.array(pil.resize((new_w, new_h), PILImage.LANCZOS))
    canvas = np.zeros((28, 28), dtype='uint8')
    py, px = (28 - new_h) // 2, (28 - new_w) // 2
    canvas[py:py + new_h, px:px + new_w] = resized
    return canvas


def rakami_tahmin_et(cizim):
    bos_olas = {str(i): 0.0 for i in range(10)}
    bos_img = PILImage.new('RGB', (280, 280), 'white')

    raw, debug = _ciz_to_uint8(cizim)
    if raw is None:
        return bos_olas, bos_img, f"⚠️ {debug}"

    img28 = _ciz_to_mnist(raw)
    if img28 is None:
        return bos_olas, bos_img, f"⚠️ Çizim algılanamadı · {debug}"

    olasiliklar = yuklenen_model.predict(
        (img28.astype('float32') / 255.0).reshape(1, 28, 28), verbose=0
    )[0]

    # Görselleştirme: 28x28 → 280x280 (10x büyütme NEAREST) + RGB (image_mode L sorun çıkarıyordu)
    display = PILImage.fromarray(img28).resize((280, 280), PILImage.NEAREST).convert('RGB')
    en_iyi = int(np.argmax(olasiliklar))
    info = f"✅ {debug} · bbox→20px center · tahmin={en_iyi} ({olasiliklar[en_iyi] * 100:.0f}%)"
    return {str(i): float(olasiliklar[i]) for i in range(10)}, display, info


# Gradio arayüzü
arayuz = gr.Interface(
    fn=rakami_tahmin_et,
    inputs=gr.Sketchpad(label="✍️ Buraya bir rakam çizin (0-9)",
                        type='numpy',
                        height=350, width=350,
                        brush=gr.Brush(default_size=22, colors=['#000000'], color_mode='fixed')),
    outputs=[
        gr.Label(num_top_classes=3, label='🎯 En Olası 3 Tahmin'),
        gr.Image(label='🔍 Modelin Gördüğü (28×28, 10× büyütüldü)',
                 width=280, height=280, show_label=True),
        gr.Textbox(label='🔧 Debug', lines=1)
    ],
    title='✍️ MEB YEĞİTEK · El Yazısı Rakam Tanıma',
    description='Beyaz canvas\'a **kalın siyah kalemle, ortaya, BÜYÜK** bir rakam (0-9) çizin. "Modelin Gördüğü" kutusunda net beyaz bir rakam görmelisiniz (10× büyütüldü).',
    flagging_mode='never'
)

print("🌐 Gradio arayüzü hazırlandı. Şimdi launch() ile başlatacağız...")

### Arayüzü Başlatalım!

`launch()` çağrısı bir web sunucusu açar:
- **Yerel makinada:** http://127.0.0.1:7860
- **Colab'da:** Otomatik tıklanabilir bağlantı verir

> 🎓 **MNIST önişleme sırrı:** Çizimi doğrudan 28×28'e küçültsek model çoğu zaman yanılır — çünkü MNIST'teki rakamlar **20×20 alana** yerleştirilmiş, etrafında **4 piksel boşluk** var. Bu yüzden tahmin fonksiyonu önce çizimin sınırlarını bulup 20'ye ölçekliyor, sonra 28×28'in ortasına koyuyor. Sağdaki "Modelin Gördüğü" kutusu bunu canlı gösterir.
>
> ⚠️ **Tahminler hâlâ yanılırsa:** "Modelin Gördüğü" panelinde NET bir rakam görüyor musun? (a) Görmüyorsan → çiziminin BÜYÜK ve KALIN olması lazım, "7" çiziyorsan üstten yatay çizgi + sağdan aşağı diyagonal — Avrupa stili çapraz çizgili 7 MNIST'te yok. (b) Görüyorsan ama tahmin yanlışsa → model basit bir Dense ağ, 5 epoch eğitildi, ileri sayfada (notebook 08) CNN ile çok daha iyisini yapacağız.

In [ ]:
arayuz.launch(share=False)   # share=True yaparsanız Gradio internette geçici link verir

## 📝 Özet

Bu notebook'ta **gerçek bir uygulama** yaptık:

| Adım | Komut |
|---|---|
| Modeli eğit | `model.fit(...)` |
| Modeli kaydet | `model.save('dosya.keras')` |
| Modeli yükle | `keras.models.load_model('dosya.keras')` |
| Web arayüzü oluştur | `gr.Interface(fn=..., inputs=..., outputs=...)` |
| Arayüzü başlat | `arayuz.launch()` |

### 🎓 Anahtar İçgörü

> **AI ürünü yapmak = Model + Arayüz + Persistans (kaydetme).**

Buraya kadar görüntülerle çalıştık ama **düz Dense katmanlarla**. Sıradaki notebook'ta görüntülere özel **Konvolüsyonel Sinir Ağları (CNN)** ile çok daha iyi sonuçlar alacağız! 🎨

<div style="background: #0B3D91; color: #BBDEFB; padding: 26px 30px; border-radius: 14px; margin-top: 32px; font-family: Georgia, serif;">

<h3 style="color: white; margin: 0 0 12px 0; border-bottom: 2px solid #FFC107; padding-bottom: 8px; display: inline-block;">MEB · YEĞİTEK</h3>

<p style="font-family: Calibri, sans-serif; font-size: 14px; color: #E3F2FD; margin: 0 0 8px 0; line-height: 1.6;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong><br>
Öğretmen ve eğitimciler için uygulamalı derin öğrenme eğitim serisi.
</p>

<p style="font-family: Calibri, sans-serif; font-size: 12px; color: #BBDEFB; margin: 12px 0 0 0;">
🎓 Bu notebook eğitim amaçlıdır · Kullanılan tüm kütüphaneler açık kaynaktır.
</p>

<div style="background: rgba(255,255,255,0.08); border-left: 3px solid #FFC107; padding: 12px 16px; margin-top: 16px; font-family: Calibri, sans-serif; font-size: 13px; color: #FFF9E6;">
➡️ <strong>Bir sonraki notebook:</strong> <code>08_cnn_giysi_siniflandirma.ipynb</code> — CNN ile Fashion-MNIST giysi sınıflandırma</div>
<p style="font-family: Calibri, sans-serif; font-size: 11px; color: #90CAF9; margin: 14px 0 0 0; text-align: center; border-top: 1px solid rgba(255,255,255,0.15); padding-top: 12px;">
© 2026 MEB YEĞİTEK · Derin Öğrenme Serisi
</p>

</div>